In [21]:
from scipy.integrate import quad
import numpy as np

In [22]:
ROUND_VALUE = 10

Objetivo: calcular los coeficientes de los filtros de paso bajo de las ondículas de Daubechies

Partimos $g_k$ y queremos encontrar un $m$ tal que $g(\xi) = |(m(\xi))|^2$ y $m$ es filtro de paso bajo. <br>
$g_{k}(\xi)$ = 1 - $c_k \cdot$  $\int_{0}^{\xi} \sin(t)^{2k+1} \, dt$ <br> <br>
con $\frac{1}{c_k}$ = $\int_{0}^{\pi} \sin(t)^{2k+1} \, dt$

In [23]:
def create_g_k(k):
    ck = 1/quad(lambda t: np.sin(t)**(2*k+1), 0, np.pi)[0]
    def g_k(w):
        return 1 - ck * quad(lambda t: np.sin(t)**(2*k+1), 0, w)[0]
    return g_k

La función $g_{k}(\xi)$ es un polinomio trigonométrico de orden $2k+1$. Hay que hallar los coeficientes de este polinomio para representarlo como suma de exponenciales complejas.

$g_k(\xi) = a_0+a_1\cdot cos(\xi) + a_2 \cdot cos(2\xi) + ... + a_{2k+1}\cdot cos((2k+1)\xi)$ <br>
$a_0 = \frac{1}{2\pi}\int_{0}^{2\pi} g(t) \, dt$ <br>
$a_k = \frac{1}{\pi}\int_{0}^{2\pi} g(t) \cdot cos(kt) \, dt$  para k > 0<br><br>

Podemos escribir $g_k(\xi)$ como $g_k(\xi) = \sum_{n=-(2k+1)}^{2k+1} b_n \cdot e^{i n \xi}$ . <br>
$cos(n\xi)$= $\frac{e^{i n \xi} + e^{-i n \xi}}{2}$ <br> 
Por tanto $b_k = b_{-k} = \frac{a_k}{2}$ para $k>0$

In [24]:
def find_bk(g, k):
    '''
    Coeficientes para expresarlo como serie de exp(ikw).
    '''

    return quad(lambda t: g(t)*np.cos(k*t),
                    0, 2*np.pi)[0] / (2*np.pi)

In [25]:
def get_coefficients_gk(k):
    
    '''
    Calcula los coeficientes de la función g_k 
    cuando se escribe como serie de exp(ikw)
    '''

    gk = create_g_k(k)

    coefs = [0]*(2*(2*k+1)+1)
    for i in range(2*k+1+1):
        cc = find_bk(gk, i)
        coefs[2*k+1+i] = cc
        coefs[2*k+1-i] = cc
    
    return coefs

In [26]:
for k in range(10):
    print(get_coefficients_gk(k))



[0.25, 0.5, 0.25]
[-0.03125000000000003, -3.122502256758253e-17, 0.28125000000000006, 0.49999999999999994, 0.28125000000000006, -3.122502256758253e-17, -0.03125000000000003]
[0.005859374999999996, -7.979727989493314e-17, -0.04882812500000005, -2.775557561562891e-17, 0.29296874999999994, 0.5, 0.29296874999999994, -2.775557561562891e-17, -0.04882812500000005, -7.979727989493314e-17, 0.005859374999999996]
[-0.0012207031249998844, -7.541565882886448e-17, 0.011962890624999953, -7.979727989493313e-17, -0.059814453125, -1.3877787807814457e-17, 0.299072265625, 0.49999999999999983, 0.299072265625, -1.3877787807814457e-17, -0.059814453125, -7.979727989493313e-17, 0.011962890624999953, -7.541565882886448e-17, -0.0012207031249998844]


[0.0002670288085938576, 1.1430118386509514e-16, -0.00308990478515612, -5.300924469105861e-17, 0.017303466796874823, -4.510281037539699e-17, -0.06729125976562501, -1.3877787807814457e-17, 0.3028106689453125, 0.5000000000000001, 0.3028106689453125, -1.3877787807814457e-17, -0.06729125976562501, -4.510281037539699e-17, 0.017303466796874823, -5.300924469105861e-17, -0.00308990478515612, 1.1430118386509514e-16, 0.0002670288085938576]
[-6.0081481933831434e-05, 8.834874115176436e-18, 0.000807762145996187, 1.1702757079907537e-16, -0.005192756652831913, -4.8591807633470396e-17, 0.021809577941894344, -7.28583859910259e-17, -0.07269859313964845, -1.0408340855860843e-17, 0.3053340911865233, 0.5000000000000001, 0.3053340911865233, -1.0408340855860843e-17, -0.07269859313964845, -7.28583859910259e-17, 0.021809577941894344, -4.8591807633470396e-17, -0.005192756652831913, 1.1702757079907537e-16, 0.000807762145996187, 8.834874115176436e-18, -6.0081481933831434e-05]
[1.3768672942836555e-05, -4.4280816141

Con los coefs de $g_k$ construimos el polinomio P(z) que es de orden $2*(2k+1)$

In [27]:
def get_roots_inside_unit_circle(coefs):
    '''
    Calcula las raíces dentro del circulo unitario
    '''

    roots = sorted(np.roots(coefs), key=abs)
    return roots[:len(roots)//2]

Hallamos las raíces de P(z) y nos quedamos con las que están dentro del círculo unidad. Con estas raíces construimos el filtro m. <br>
Los coeficientes de $m(\xi)$ los calculamos a partir de sus raíces. <br>

In [28]:
def get_filter_coefficients(roots):
    '''
    Calcula los coeficientes del filtro a partir de las raices
    h_0 = (-1)^n * r1*r2*...*rn
    ....
    h_(n-1) = -(r1 + r2 + ... + rn)
    h_n = 1

    '''

    coefs = [1]
    for r in roots:
        coefs = np.convolve(coefs, [1, -r])

    f = np.sqrt(2)/sum(coefs)

    #Normalizar 
    return [f*c for c in coefs]

In [38]:
def get_Daubechies_filter(k):
    coefs = get_coefficients_gk(k)
    print("COEFS\n")
    print(coefs)
    roots = get_roots_inside_unit_circle(coefs)
    print("\nROOTS\n")
    print(roots)
    print("\n")
    filter_coefs = get_filter_coefficients(roots)

    # Los coeficientes han de ser reales. Forzamos la parte imaginaria a 0
    # para contrarestar errores de redondeo

    filter_coefs = np.round(np.real(filter_coefs), ROUND_VALUE)
    return filter_coefs

In [35]:
# imprimimos valores de los filtros. 
# (k+1 para seguir con la notación de la tabla que aparece en Daubechies Ten Lectures On Wavelets)
for k in range(1, 10):
    print(k+1, get_Daubechies_filter(k))

[-0.03125000000000003, -3.122502256758253e-17, 0.28125000000000006, 0.49999999999999994, 0.28125000000000006, -3.122502256758253e-17, -0.03125000000000003]
[(0.26794919243112275+0j), (-0.9997937464333604+0j), (-0.9999999789266865+0.00020627463982794851j)]
2 [ 0.48301272  0.83650296  0.22409406 -0.12939618]
[0.005859374999999996, -7.979727989493314e-17, -0.04882812500000005, -2.775557561562891e-17, 0.29296874999999994, 0.5, 0.29296874999999994, -2.775557561562891e-17, -0.04882812500000005, -7.979727989493314e-17, 0.005859374999999996]
[(0.2872513780440208+0.1528923338821984j), (0.2872513780440208-0.1528923338821984j), (-0.9968373076632404+0j), (-0.9984133720731845+0.002738990793517797j), (-0.9984133720731845-0.002738990793517797j)]
3 [ 0.33372612  0.8073373   0.45832921 -0.13534541 -0.08494854  0.03511487]
[-0.0012207031249998844, -7.541565882886448e-17, 0.011962890624999953, -7.979727989493313e-17, -0.059814453125, -1.3877787807814457e-17, 0.299072265625, 0.49999999999999983, 0.2990722

In [41]:
print(get_Daubechies_filter(2))

COEFS

[0.005859374999999996, -7.979727989493314e-17, -0.04882812500000005, -2.775557561562891e-17, 0.29296874999999994, 0.5, 0.29296874999999994, -2.775557561562891e-17, -0.04882812500000005, -7.979727989493314e-17, 0.005859374999999996]

ROOTS

[(0.2872513780440208+0.1528923338821984j), (0.2872513780440208-0.1528923338821984j), (-0.9968373076632404+0j), (-0.9984133720731845+0.002738990793517797j), (-0.9984133720731845-0.002738990793517797j)]


[ 0.33372612  0.8073373   0.45832921 -0.13534541 -0.08494854  0.03511487]


In [31]:
# Para k = 3

